# MOVIN video compression for GitHub upload\n\nUse this notebook in Google Colab to compress the large MOVIN screen recording zip into a small `source_video.zip` that can be uploaded through the GitHub web UI.\n\nRecommended default: keep the original video duration and narration timing, target ~22 MB, and let the GitHub workflow later create the final 6-minute Ryan Neural narrated video.

In [ ]:
# 1) Install FFmpeg\n!apt-get update -qq > /dev/null\n!apt-get install -y -qq ffmpeg > /dev/null\n!ffmpeg -version | head -1

## 2) Upload the large zip/video\n\nRun the cell below and upload your file, for example `Screen Recording 2026-07-06 142038.zip`. Colab can handle this better than GitHub web upload.

In [ ]:
from google.colab import files\nfrom pathlib import Path\nuploaded = files.upload()\nINPUT_PATH = Path(next(iter(uploaded.keys()))).resolve()\nprint('Uploaded:', INPUT_PATH, 'size MB=', INPUT_PATH.stat().st_size/1024/1024)

### Alternative: use Google Drive instead of browser upload\nUncomment and run this cell if the browser upload is unstable.

In [ ]:
# from google.colab import drive\n# from pathlib import Path\n# drive.mount('/content/drive')\n# INPUT_PATH = Path('/content/drive/MyDrive/Screen Recording 2026-07-06 142038.zip')\n# print('Using Drive file:', INPUT_PATH, 'size MB=', INPUT_PATH.stat().st_size/1024/1024)

## 3) Create the compression script in Colab\nThis writes the same script included in the repo to `/content/compress_video_for_github.py`.

In [ ]:
%%writefile /content/compress_video_for_github.py\n#!/usr/bin/env python3\nfrom __future__ import annotations\nimport argparse, json, os, shutil, subprocess, sys, zipfile\nfrom pathlib import Path\nVIDEO_EXTS = ('.mp4','.mov','.mkv','.webm','.avi','.m4v')\ndef run(cmd):\n    print('\\n$',' '.join(map(str,cmd)), flush=True); return subprocess.run(cmd, check=True)\ndef cap(cmd): return subprocess.check_output(cmd, text=True).strip()\ndef duration(p): return float(cap(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)]))\ndef has_audio(p):\n    try: return bool(cap(['ffprobe','-v','error','-select_streams','a','-show_entries','stream=index','-of','csv=p=0',str(p)]).strip())\n    except Exception: return False\ndef atempo(f):\n    parts=[]; r=f\n    while r>2.0: parts.append(2.0); r/=2.0\n    while r<0.5: parts.append(0.5); r/=0.5\n    parts.append(r); return ','.join(f'atempo={x:.6f}' for x in parts)\ndef extract(inp, out):\n    if inp.suffix.lower() in VIDEO_EXTS: return inp\n    d=out/'extracted'; d.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(inp) as z:\n        vids=[m for m in z.namelist() if m.lower().endswith(VIDEO_EXTS) and not m.endswith('/')]\n        if not vids: raise FileNotFoundError('No video inside zip')\n        m=sorted(vids, key=lambda x:z.getinfo(x).file_size, reverse=True)[0]\n        print('Extracting',m); z.extract(m,d); return d/m\ndef zipit(mp4,zp):\n    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as z: z.write(mp4, arcname='source_video.mp4')\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument('--input',required=True); ap.add_argument('--output-dir',default='/content/movin_compressed'); ap.add_argument('--target-mb',type=float,default=22); ap.add_argument('--height',type=int,default=720); ap.add_argument('--fps',type=int,default=24); ap.add_argument('--audio-kbps',type=int,default=48); ap.add_argument('--speed-to-seconds',type=float,default=0)\n    a=ap.parse_args(); inp=Path(a.input).resolve(); out=Path(a.output_dir); out.mkdir(parents=True,exist_ok=True); work=out/'_work'; work.mkdir(exist_ok=True)\n    src=extract(inp,work); dur=duration(src); bitrate_dur=a.speed_to_seconds if a.speed_to_seconds>0 else dur\n    print(f'Source duration: {dur:.1f}s | source size: {src.stat().st_size/1024/1024:.1f} MB')\n    mp4=out/'source_video.mp4'; zp=out/'source_video.zip'; attempts=[]; height=a.height\n    for attempt in range(1,5):\n        safety=0.86-(attempt-1)*0.04; total_kbits=a.target_mb*8192*safety; vkbps=max(80,int(total_kbits/max(1,bitrate_dur)-a.audio_kbps))\n        vf=[f'scale=-2:min({height}\\,ih)',f'fps={a.fps}','format=yuv420p']; af=[]\n        if a.speed_to_seconds>0 and abs(dur-a.speed_to_seconds)>1:\n            speed=dur/a.speed_to_seconds; vf.insert(0,f'setpts=PTS/{speed:.8f}'); af=[atempo(speed)] if has_audio(src) else []; print(f'Speeding {dur:.1f}s -> {a.speed_to_seconds:.1f}s')\n        vfilt=','.join(vf); passlog=str(out/'ffmpeg2pass')\n        run(['ffmpeg','-y','-i',str(src),'-vf',vfilt,'-c:v','libx264','-preset','slow','-b:v',f'{vkbps}k','-pass','1','-passlogfile',passlog,'-an','-f','mp4','/dev/null'])\n        cmd=['ffmpeg','-y','-i',str(src),'-vf',vfilt,'-c:v','libx264','-preset','slow','-b:v',f'{vkbps}k','-pass','2','-passlogfile',passlog]\n        if has_audio(src):\n            if af: cmd += ['-af', ','.join(af)]\n            cmd += ['-c:a','aac','-b:a',f'{a.audio_kbps}k']\n        else: cmd += ['-an']\n        cmd += ['-movflags','+faststart',str(mp4)]; run(cmd)\n        for p in out.glob('ffmpeg2pass*'): p.unlink(missing_ok=True)\n        zipit(mp4,zp); zmb=zp.stat().st_size/1024/1024; mmb=mp4.stat().st_size/1024/1024\n        attempts.append({'attempt':attempt,'height':height,'video_kbps':vkbps,'mp4_mb':round(mmb,3),'zip_mb':round(zmb,3)})\n        print(f'Attempt {attempt}: MP4={mmb:.2f} MB ZIP={zmb:.2f} MB target={a.target_mb} MB')\n        if zmb <= a.target_mb: break\n        height=max(360,int(height*.75)//2*2)\n    manifest={'input':str(inp),'source_duration_seconds':round(dur,3),'target_mb':a.target_mb,'final_mp4':str(mp4),'final_zip':str(zp),'final_zip_mb':round(zp.stat().st_size/1024/1024,3),'attempts':attempts,'next_step':'Upload source_video.zip to input/source_video.zip in the GitHub repo.'}\n    (out/'compression_manifest.json').write_text(json.dumps(manifest,indent=2)); print('DONE:', zp)\nif __name__=='__main__': main()

## 4) Compress for GitHub upload\n\nDefault keeps the original duration and audio timing so the GitHub workflow can transcribe/revoice the same narration. It targets ~22 MB so it can be uploaded through the GitHub web UI.

In [ ]:
TARGET_MB = 22      # Keep <= 24 MB for GitHub web upload comfort\nHEIGHT = 720        # Use 540 or 480 if you need a smaller file\nFPS = 24\nAUDIO_KBPS = 48     # Good enough for speech transcription\nSPEED_TO_SECONDS = 0  # Recommended: 0 preserves original narration timing. Use 360 only for a six-minute preview.\n\n!python /content/compress_video_for_github.py \\n  --input "{INPUT_PATH}" \\n  --output-dir /content/movin_compressed \\n  --target-mb {TARGET_MB} \\n  --height {HEIGHT} \\n  --fps {FPS} \\n  --audio-kbps {AUDIO_KBPS} \\n  --speed-to-seconds {SPEED_TO_SECONDS}

## 5) Download the compressed files\nUpload `source_video.zip` into your GitHub repo at `input/source_video.zip`, then run the Ryan Neural workflow.

In [ ]:
from google.colab import files\nfiles.download('/content/movin_compressed/source_video.zip')\nfiles.download('/content/movin_compressed/compression_manifest.json')